# Kiran's Local Agent Loop for code reviews on GitHub 

In [59]:
from rich.console import Console
import os, json, re, requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

True

In [60]:
console = Console()
def show(text):
    try:
        console.print(text)
    except Exception:
        print(text)

In [73]:
# Choose the model provider: "anthropic" (cloud) or "ollama" (local)
PROVIDER = "anthropic"

if PROVIDER == "ollama":
    # Ollama serves an OpenAI-compatible API; the api_key is required by the client but ignored.
    OLLAMA_URL = "http://localhost:11434"
    BASE_MODEL = "qwen3-coder:30b"
    NUM_CTX = 32_768  # Ollama defaults to 4096 tokens, too small for a diff plus tool history
    MODEL = f"{BASE_MODEL.split(':')[0]}-{NUM_CTX // 1024}k"
    # The OpenAI-compatible endpoint can't set num_ctx, so create a variant with a larger
    # context window (reuses the same weights; nothing is downloaded).
    r = requests.post(f"{OLLAMA_URL}/api/create",
                      json={"model": MODEL, "from": BASE_MODEL,
                            "parameters": {"num_ctx": NUM_CTX}, "stream": False})
    r.raise_for_status()
    # The api_key is required by the client but ignored by Ollama.
    client = OpenAI(api_key="ollama", base_url=f"{OLLAMA_URL}/v1")
    MAX_DIFF_CHARS = 60_000  # ~15k tokens, leaves room for prompt, tool history and the review
elif PROVIDER == "anthropic":
    client = OpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url="https://api.anthropic.com/v1/")
    MODEL = "claude-sonnet-5-5"
    MAX_DIFF_CHARS = 80_000
else:
    raise ValueError(f"Unknown PROVIDER: {PROVIDER}")

show(f"[cyan]Using[/cyan] {PROVIDER} / {MODEL}")

Using anthropic / claude-sonnet-5-5

In [62]:
# Github as Source for Pull Requests
GH = "https://api.github.com"
TOKEN = os.getenv("GITHUB_TOKEN")
HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
# Only send auth when a token exists; "Bearer None" makes GitHub return 401 even for public repos.
if TOKEN:
    HEADERS["Authorization"] = f"Bearer {TOKEN}"

In [63]:
# Read the PR from Pull request link to prepare information about Pull requests
 
def parse_pr(pr: str) -> tuple[str, str, int]:
    """Accept a URL or 'owner/repo#123'."""
    m = re.search(r"github\.com/([^/]+)/([^/]+)/pull/(\d+)", pr)
    if m:
        return m.group(1), m.group(2), int(m.group(3))
    m = re.match(r"([^/]+)/([^/#]+)[#/](\d+)", pr)
    if m:
        return m.group(1), m.group(2), int(m.group(3))
    raise ValueError(f"Could not parse PR: {pr}")

def get_pr(pr: str) -> str:
    owner, repo, number = parse_pr(pr)
    r = requests.get(f"{GH}/repos/{owner}/{repo}/pulls/{number}", headers=HEADERS)
    r.raise_for_status()
    data = r.json()
    slim = {
        "title": data["title"],
        "body": data.get("body") or "",
        "user": data["user"]["login"],
        "base": data["base"]["ref"],
        "head": data["head"]["ref"],
        "additions": data["additions"],
        "deletions": data["deletions"],
        "changed_files": data["changed_files"],
        "html_url": data["html_url"],
    }
    show(f"[cyan]Fetched PR[/cyan] {slim['title']}")
    return json.dumps(slim, indent=2)

In [64]:
# Get the PR diff and pr files to be reviewed

def get_pr_diff(pr: str) -> str:
    owner, repo, number = parse_pr(pr)
    headers = {**HEADERS, "Accept": "application/vnd.github.diff"}
    r = requests.get(f"{GH}/repos/{owner}/{repo}/pulls/{number}", headers=headers)
    r.raise_for_status()
    diff = r.text
    # Keep the loop small; huge PRs blow the context window.
    if len(diff) > MAX_DIFF_CHARS:
        diff = diff[:MAX_DIFF_CHARS] + "\n\n[truncated]"
    show(f"[cyan]Fetched diff[/cyan] ({len(diff)} chars)")
    return diff
    
def get_pr_files(pr: str) -> str:
    owner, repo, number = parse_pr(pr)
    r = requests.get(f"{GH}/repos/{owner}/{repo}/pulls/{number}/files",
                     headers=HEADERS, params={"per_page": 100})
    r.raise_for_status()
    files = [
        {"filename": f["filename"], "status": f["status"],
         "additions": f["additions"], "deletions": f["deletions"]}
        for f in r.json()
    ]
    return json.dumps(files, indent=2)

In [65]:
def post_pr_comment(pr: str, body: str) -> str:
    """Skip this until the printed review looks good."""
    owner, repo, number = parse_pr(pr)
    r = requests.post(
        f"{GH}/repos/{owner}/{repo}/issues/{number}/comments",
        headers=HEADERS,
        json={"body": body},
    )
    r.raise_for_status()
    show("[green]Posted comment[/green]")
    return r.json()["html_url"]

In [66]:
# Some lists!

checklist = []
completed = []

def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

def create_checklist(descriptions: list[str]) -> str:
    checklist.clear()
    completed.clear()   
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [67]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    show(completion_notes)
    return get_checklist_report()

In [68]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [69]:
get_pr_json = {
    "name": "get_pr",
    "description": "Fetch title, body, author, and stats for a GitHub pull request.",
    "parameters": {
        "type": "object",
        "properties": {"pr": {"type": "string", "description": "PR URL or owner/repo#number"}},
        "required": ["pr"],
        "additionalProperties": False,
    },
}
get_pr_diff_json = {
    "name": "get_pr_diff",
    "description": "Fetch the unified diff for a GitHub pull request.",
    "parameters": {
        "type": "object",
        "properties": {"pr": {"type": "string"}},
        "required": ["pr"],
        "additionalProperties": False,
    },
}
get_pr_files_json = {
    "name": "get_pr_files",
    "description": "List changed files in a GitHub pull request.",
    "parameters": {
        "type": "object",
        "properties": {"pr": {"type": "string"}},
        "required": ["pr"],
        "additionalProperties": False,
    },
}
post_pr_comment_json = {
    "name": "post_pr_comment",
    "description": "Post a comment on a GitHub pull request and return the comment URL. Call this once, with the final review.",
    "parameters": {
        "type": "object",
        "properties": {
            "pr": {"type": "string", "description": "PR URL or owner/repo#number"},
            "body": {"type": "string", "description": "The comment text in GitHub-flavored Markdown"},
        },
        "required": ["pr", "body"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": create_checklist_json},
    {"type": "function", "function": mark_complete_json},
    {"type": "function", "function": get_pr_json},
    {"type": "function", "function": get_pr_diff_json},
    {"type": "function", "function": get_pr_files_json},
    {"type": "function", "function": post_pr_comment_json},
]

In [70]:
TOOL_FUNCS = {f.__name__: f for f in
              [create_checklist, mark_complete, get_pr, get_pr_diff, get_pr_files, post_pr_comment]}

def handle_tool_calls(tool_calls):
    results = []
    for tc in tool_calls:
        fn = TOOL_FUNCS.get(tc.function.name)
        try:
            args = json.loads(tc.function.arguments or "{}")
            result = fn(**args) if fn else f"Error: unknown tool {tc.function.name}"
        except Exception as e:
            result = f"Error: {type(e).__name__}: {e}"
        content = result if isinstance(result, str) else json.dumps(result)
        results.append({"role": "tool", "content": content, "tool_call_id": tc.id})
    return results

In [71]:
def loop(messages, max_turns=12):
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )
    turns = 0
    while response.choices[0].finish_reason == "tool_calls" and turns < max_turns:
        message = response.choices[0].message
        results = handle_tool_calls(message.tool_calls)
        messages.append(message.model_dump(exclude_none=True))
        messages.extend(results)
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )
        turns += 1
    content = response.choices[0].message.content
    if response.choices[0].finish_reason == "tool_calls":
        content = f"[red]Stopped after {max_turns} turns without a final answer.[/red]"
    show(content or "[red]No content returned.[/red]")

In [ ]:
system_message = """
You are a senior engineer reviewing a GitHub pull request.

Use tools. Do not guess the diff.
1. create_checklist with steps: fetch PR, list files, fetch diff, write findings, post comment.
2. Call get_pr, get_pr_files, get_pr_diff.
3. Review for bugs, security, tests, API contracts, and missing edge cases.
4. mark_complete as you go.
5. Write the review in this shape:
   - Summary
   - What looks good
   - Findings (severity: high/medium/low, file + line if possible)
   - Questions for the author
6. Call post_pr_comment exactly once with the review in GitHub Markdown, then mark
   the last item complete.
7. Finally, reply with the same review in Rich markup (no fenced code blocks).
Do not ask the user questions.
"""

user_message = "Review https://github.com/<owner>/<reponame>/pull/<PR number>"
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]
loop(messages)

Checklist #1: Fetch PR
Checklist #2: List files
Checklist #3: Fetch diff
Checklist #4: Write findings
Checklist #5: Post comment

Fetched PR Bitbucket server support

Fetched diff (28965 chars)

Fetched PR metadata (3 files, +748/-0, empty description).

Checklist #1: Fetch PR
Checklist #2: List files
Checklist #3: Fetch diff
Checklist #4: Write findings
Checklist #5: Post comment

Listed files: .env.example, PR_URL_Review_React_Chunked.py, pr_chunking_utils.py (all new).

... (5 outputs left out)

Posted comment

Posted review: https://github.com/kkiran33/automated_code_reviews/pull/4#issuecomment-6721069128

Checklist #1: Fetch PR
Checklist #2: List files
Checklist #3: Fetch diff
Checklist #4: Write findings
Checklist #5: Post comment

Review: Bitbucket Server support (PR #4)
Posted at: https://github.com/kkiran33/automated_code_reviews/pull/4#issuecomment-6721069128

Summary
The PR adds a three-phase chunked review pipeline (overview, per-chunk review, synthesis) and a `.env.example`. It 
also adds Bitbucket Server REST 1.0 support alongside GitHub and Bitbucket Cloud. The script won't run as written, 
and the per-chunk prompt never includes the code to review. The Bitbucket Server path, which is the feature in the 
PR title, also has several correctness problems. There are no tests. Recommendation: request changes.

What looks good
- The code is split cleanly into provider adapters, a model layer and chunking utilities.
- Every HTTP call has a timeout.
- Bitbucket Server URL parsing handles context paths.
- Failed chunks are isolated, and synthesis only uses the reviews that succeeded.
- There is a `--dry-run` mode.

Findings

High
1. `typing` is never imported (`Tuple`, `List`), so the module raises `NameError` at import time.
2. `call_local_model` is annotated as returning 2 values but returns 3.
3. `create_file_review_prompt` never includes `chunk.content`, and the overview prompt only sends file names and 
line counts. The model reviews nothing, and the hallucinated output is posted to the PR.
4. `parts[3].lstrip('b/')` strips a character set, not a prefix. `b/backend.py` becomes `ackend.py`. Use 
`removeprefix`.
5. The Bitbucket Server `/diff` endpoint returns JSON by default, not unified diff text. `parse_diff_by_file` then 
finds no files, and an empty review is posted.
6. `--auto-approve` and the `approve_*` functions are dead code. Please remove them or implement them deliberately,
since auto-approving on LLM output is risky.

Medium
7. The env var names in `.env.example` don't match the ones the code reads. `MODEL_API`, `USE_LARGE_MODEL_PHASE_3` 
and `BITBUCKET_SERVER_URL` are undocumented. Bitbucket Server usually uses a PAT with a Bearer header.
8. `detect_provider` relies on the loose check `"bitbucket" in host`. Compare against the configured server URL 
instead.
9. The Bitbucket Server branch names use `repository.name` instead of `displayId`.
10. `process_chunks_parallel` waits on futures sequentially, so one hung call blocks the rest, and the timeout 
doesn't cancel the thread. Use `as_completed` and sort by `chunk_id`.
11. Token estimation is whitespace-based and under-counts code tokens. Splitting loses hunk headers. 
`ChunkingConfig`, `chunk_overlap`, `start_index` and `end_index` are unused.
12. Empty diffs or skipped phases still post a blank report.
13. Failures are swallowed, and "Review posted successfully!" prints even when posting failed.
14. Bitbucket Cloud `description` can be `None`, and there is no `or ""` guard. Auth headers may be dropped on the 
diff redirect.
15. There is no truncation or splitting for long comments.

Low
16. `main()` exits with code 0 on error.
17. Error messages include `response.text`, which may leak server content.
18. Credentials go over plain HTTP by default, with no warning.
19. Prompt injection from PR content: the diff should be labelled untrusted.
20. Style:
   - "React" in the name is misleading.
   - `chr(10)` in f-strings is hard to read.
   - The `<placeholder>` integers in `.env.example` break `int()` parsing.
21. The PR description is empty.

Tests
There are none. Please add tests for `parse_diff_by_file`, `chunk_diff_by_size`, the Bitbucket Server URL parser 
and `detect_provider`. Please also add mocked-HTTP tests for the Bitbucket Server adapter.

Questions for the author
1. Has this been run end to end against a real Bitbucket Server? Which version, and what does the `/diff` response 
look like?
2. Was omitting `chunk.content` from the prompts intentional?
3. Are `ChunkingConfig` and `--auto-approve` planned for a follow-up?
4. Should Bitbucket Server authenticate with a PAT (Bearer) instead of a username and password?
5. Which env var names are the intended ones: the ones in `.env.exa